# Geo-Fraud Detection for Moving Crews
### MoveUp · HR AI prototype

Field crews check in from a phone. This notebook detects the GPS patterns that
matter to a moving company:

- impossible travel (teleporting between jobs)
- buddy punching / identical coordinates
- off-site check-ins
- GPS accuracy anomalies
- overstays at non-client locations (shown as **red zones**)
- route detours (the long way round)
- crew separation (team members far apart during a job)

The data is **synthetic** and deterministic, so every result is reproducible.
The interactive map uses real Tunis coordinates and client-style job sites.


## 0. Kaggle setup

1. Open the notebook on Kaggle and choose **Run all**.
2. Internet is only needed for the Folium basemap tiles. The detection code and
   the static map work without it.
3. `%pip install -q folium` is safe even when Folium is already preinstalled.

All outputs are written to `/kaggle/working/geo_fraud_output/`.


In [ ]:
%pip install -q folium

## 0. Imports and configuration

In [ ]:
import json
import math
import warnings
from pathlib import Path
from datetime import timedelta

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from matplotlib.lines import Line2D
from matplotlib.patches import Circle

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "geo_fraud_output"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print("Output folder:", OUT)
print("Random seed:", SEED)

## 1. The scenario: one moving-company day in Greater Tunis

We simulate 8 movers, 4 client jobs, and one working day (08:00–13:00).
Each employee has a home base, an assigned crew, and a client site. The routes
are noisy enough to look like phone GPS, with normal road-like curves.

The day contains **12 injected anomalies** across the seven detection types.
It also contains a legitimate 12-minute coffee stop. That stop is a hard
negative: a 10-minute dwell rule would flag it, so the chosen policy uses
25 minutes.


In [ ]:
DAY = pd.Timestamp("2026-06-15")

SITES = {
    "JOB-101": {"client": "Ben Salah", "area": "La Marsa", "lat": 36.8782, "lng": 10.3247},
    "JOB-102": {"client": "Trabelsi", "area": "Carthage", "lat": 36.8528, "lng": 10.3233},
    "JOB-103": {"client": "Gharbi", "area": "Tunis Centre", "lat": 36.8065, "lng": 10.1815},
    "JOB-104": {"client": "Mansour", "area": "Ariana", "lat": 36.8625, "lng": 10.1956},
}

JOBS = pd.DataFrame(
    [
        {
            "job_id": job_id,
            "client": site["client"],
            "area": site["area"],
            "lat": site["lat"],
            "lng": site["lng"],
            "start": DAY + pd.Timedelta(hours=9),
            "end": DAY + pd.Timedelta(hours=13),
        }
        for job_id, site in SITES.items()
    ]
)

EMPLOYEES = pd.DataFrame(
    [
        {"employee_id": "EMP001", "name": "Alice Johnson", "home_lat": 36.8500, "home_lng": 10.2700, "job_id": "JOB-101"},
        {"employee_id": "EMP002", "name": "Bob Smith", "home_lat": 36.8900, "home_lng": 10.2800, "job_id": "JOB-101"},
        {"employee_id": "EMP003", "name": "Carol Davis", "home_lat": 36.8400, "home_lng": 10.3000, "job_id": "JOB-102"},
        {"employee_id": "EMP004", "name": "Dave Wilson", "home_lat": 36.8300, "home_lng": 10.2900, "job_id": "JOB-102"},
        {"employee_id": "EMP005", "name": "Eve Brown", "home_lat": 36.7900, "home_lng": 10.2000, "job_id": "JOB-103"},
        {"employee_id": "EMP006", "name": "Frank Miller", "home_lat": 36.7400, "home_lng": 10.2300, "job_id": "JOB-103"},
        {"employee_id": "EMP007", "name": "Grace Lee", "home_lat": 36.8700, "home_lng": 10.1800, "job_id": "JOB-104"},
        {"employee_id": "EMP008", "name": "Henry Kim", "home_lat": 36.8800, "home_lng": 10.2100, "job_id": "JOB-104"},
    ]
)

CREWS = EMPLOYEES.groupby("job_id")["employee_id"].apply(list).to_dict()
EMP_BY_ID = EMPLOYEES.set_index("employee_id")
JOB_BY_ID = JOBS.set_index("job_id")

print("\nJobs")
display(JOBS[["job_id", "client", "area", "lat", "lng"]])
print("\nEmployees and crews")
display(EMPLOYEES[["employee_id", "name", "job_id"]].merge(
    JOBS[["job_id", "area"]], on="job_id", how="left"
))

## 2. Geometry helpers

`haversine_m` converts latitude/longitude pairs to metres. The synthetic legs
are interpolated with a small lateral bow so a normal route is not a perfectly
straight line. GPS noise is added in metres, not degrees.


In [ ]:
def haversine_m(lat1, lng1, lat2, lng2):
    """Great-circle distance in metres."""
    radius = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * radius * math.asin(math.sqrt(a))


def point_at(start, end, fraction, perp_offset_deg=0.0):
    """Linear interpolation between two (lat, lng) points with a lateral offset."""
    lat1, lng1 = start
    lat2, lng2 = end
    dlat, dlng = lat2 - lat1, lng2 - lng1
    length = math.hypot(dlat, dlng) or 1.0
    perp_lat, perp_lng = -dlng / length, dlat / length

    lat = lat1 + dlat * fraction + perp_lat * perp_offset_deg
    lng = lng1 + dlng * fraction + perp_lng * perp_offset_deg
    return lat, lng


def jitter(point, metres):
    lat, lng = point
    return (
        lat + rng.normal(0, metres) / 111_320,
        lng + rng.normal(0, metres) / (111_320 * math.cos(math.radians(lat))),
    )


def leg_pings(start, end, t0, t1, *, wiggle_deg=0.0015, noise_m=7.0, interval_min=2):
    """GPS pings along one leg, spaced by `interval_min` minutes."""
    times = pd.date_range(t0, t1, freq=f"{interval_min}min")
    if len(times) == 0 or times[-1] < t1:
        times = times.append(pd.DatetimeIndex([t1]))

    rows = []
    n = max(len(times), 2)
    for i, timestamp in enumerate(times):
        fraction = i / (n - 1)
        # A smooth bow makes a normal road-like path, not a straight teleport.
        offset = wiggle_deg * math.sin(math.pi * fraction) if start != end else 0.0
        lat, lng = point_at(start, end, fraction, offset)
        lat, lng = jitter((lat, lng), noise_m)
        rows.append({"timestamp": timestamp, "lat": lat, "lng": lng})
    return rows

## 3. Synthetic GPS traces with labelled anomalies

Each employee gets a home-to-job leg and an on-site dwell. Specific employees
then receive an injected event:

| Employee | Injected behaviour |
|---|---|
| EMP001 | legitimate 12-minute coffee stop (hard negative) |
| EMP002 | leaves the La Marsa job for Gammarth (crew separation + overstay) |
| EMP003 | checks in ~800 m from the Carthage job |
| EMP004 | 40-minute café overstay 1.8 km from the Carthage job |
| EMP005 | 12 minutes of 250 m GPS accuracy |
| EMP006 | detour via La Goulette (22 km for an 8.5 km trip) |
| EMP007 | checks in at another crew's job (buddy punch) |
| EMP008 | 26 km round-trip teleport in 4 minutes |


In [ ]:
def base_day(employee):
    home = (employee.home_lat, employee.home_lng)
    job = tuple(JOB_BY_ID.loc[employee.job_id, ["lat", "lng"]].astype(float))
    pings = []
    pings += leg_pings(home, job, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50), wiggle_deg=0.0018)
    pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=13), wiggle_deg=0.0, noise_m=11.0)
    return pings, home, job


def trace_for(employee):
    """Return (pings, scenario) for one employee. Anomalies are injected here."""
    emp_id = employee.employee_id
    pings, home, job = base_day(employee)

    if emp_id == "EMP001":
        # Legitimate hard negative: a 12-minute coffee stop on the way to work.
        # The 25-minute dwell threshold must ignore this; a 10-minute threshold would not.
        cafe = (36.8620, 10.2950)
        pings = []
        pings += leg_pings(home, cafe, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=20))
        pings += leg_pings(cafe, cafe, DAY + pd.Timedelta(hours=8, minutes=20), DAY + pd.Timedelta(hours=8, minutes=32), wiggle_deg=0.0, noise_m=8)
        pings += leg_pings(cafe, job, DAY + pd.Timedelta(hours=8, minutes=32), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "normal_legit_stop"

    if emp_id == "EMP004":
        # Overstay: 40 minutes at a café 1.8 km from the client's house.
        cafe = (36.8485, 10.3040)
        pings = []
        pings += leg_pings(home, job, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=10), noise_m=11)
        pings += leg_pings(job, cafe, DAY + pd.Timedelta(hours=10), DAY + pd.Timedelta(hours=10, minutes=6))
        pings += leg_pings(cafe, cafe, DAY + pd.Timedelta(hours=10, minutes=6), DAY + pd.Timedelta(hours=10, minutes=46), wiggle_deg=0.0, noise_m=8)
        pings += leg_pings(cafe, job, DAY + pd.Timedelta(hours=10, minutes=46), DAY + pd.Timedelta(hours=10, minutes=52))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=10, minutes=52), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "overstay"

    if emp_id == "EMP002":
        # Crew separation: Bob leaves the job for Gammarth for ~30 minutes.
        gammarth = (36.9186, 10.2892)
        pings = []
        pings += leg_pings(home, job, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=10, minutes=10), noise_m=11)
        pings += leg_pings(job, gammarth, DAY + pd.Timedelta(hours=10, minutes=10), DAY + pd.Timedelta(hours=10, minutes=22))
        pings += leg_pings(gammarth, gammarth, DAY + pd.Timedelta(hours=10, minutes=22), DAY + pd.Timedelta(hours=10, minutes=52), wiggle_deg=0.0, noise_m=9)
        pings += leg_pings(gammarth, job, DAY + pd.Timedelta(hours=10, minutes=52), DAY + pd.Timedelta(hours=11, minutes=4))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=11, minutes=4), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "crew_separation"

    if emp_id == "EMP006":
        # Detour: Frank drives via La Goulette instead of the direct route.
        la_goulette = (36.8181, 10.3050)
        pings = []
        pings += leg_pings(home, la_goulette, DAY + pd.Timedelta(hours=7, minutes=45), DAY + pd.Timedelta(hours=8, minutes=10))
        pings += leg_pings(la_goulette, job, DAY + pd.Timedelta(hours=8, minutes=10), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=13), wiggle_deg=0.0, noise_m=11)
        return pings, "detour"

    if emp_id == "EMP008":
        # Impossible travel: a 17 km jump to Sidi Bou Said in two minutes.
        sidi_bou_said = (36.8708, 10.3417)
        pings = []
        pings += leg_pings(home, job, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=10), noise_m=11)
        pings.append({"timestamp": DAY + pd.Timedelta(hours=10, minutes=2), "lat": sidi_bou_said[0], "lng": sidi_bou_said[1]})
        pings.append({"timestamp": DAY + pd.Timedelta(hours=10, minutes=4), "lat": job[0], "lng": job[1]})
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=10, minutes=6), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "impossible_travel"

    if emp_id == "EMP003":
        # Off-site check-in: first ping is a deterministic ~800 m from the client's house.
        offset = (job[0] + 800 / 111_320, job[1])
        pings = []
        pings += leg_pings(home, offset, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(offset, job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=8, minutes=54))
        pings += leg_pings(job, job, DAY + pd.Timedelta(hours=8, minutes=54), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "off_site_checkin"

    if emp_id == "EMP007":
        # Buddy punch: Grace checks in at another crew's job, then goes to her own.
        other_job = tuple(JOB_BY_ID.loc["JOB-103", ["lat", "lng"]].astype(float))
        own_job = tuple(JOB_BY_ID.loc["JOB-104", ["lat", "lng"]].astype(float))
        pings = []
        pings += leg_pings(home, other_job, DAY + pd.Timedelta(hours=8), DAY + pd.Timedelta(hours=8, minutes=50))
        pings += leg_pings(other_job, other_job, DAY + pd.Timedelta(hours=8, minutes=50), DAY + pd.Timedelta(hours=9, minutes=30), wiggle_deg=0.0, noise_m=10)
        pings += leg_pings(other_job, own_job, DAY + pd.Timedelta(hours=9, minutes=30), DAY + pd.Timedelta(hours=9, minutes=55))
        pings += leg_pings(own_job, own_job, DAY + pd.Timedelta(hours=9, minutes=55), DAY + pd.Timedelta(hours=13), noise_m=11)
        return pings, "buddy_punch"

    if emp_id == "EMP005":
        # GPS accuracy anomaly: 12 minutes of very poor fixes.
        updated = []
        for ping in pings:
            if DAY + pd.Timedelta(hours=11) <= ping["timestamp"] <= DAY + pd.Timedelta(hours=11, minutes=12):
                lat, lng = jitter((ping["lat"], ping["lng"]), 120)
                updated.append({**ping, "lat": lat, "lng": lng, "accuracy_m": 250.0})
            else:
                updated.append(ping)
        return updated, "gps_accuracy"

    return pings, "normal"


rows = []
scenario_by_employee = {}
for _, employee in EMPLOYEES.iterrows():
    employee_pings, scenario = trace_for(employee)
    scenario_by_employee[employee.employee_id] = scenario
    for ping in employee_pings:
        rows.append(
            {
                "employee_id": employee.employee_id,
                "employee_name": employee["name"],
                "job_id": employee.job_id,
                "timestamp": ping["timestamp"],
                "lat": ping["lat"],
                "lng": ping["lng"],
                "accuracy_m": ping.get("accuracy_m", float(rng.uniform(8, 22))),
            }
        )

pings = (
    pd.DataFrame(rows)
    .drop_duplicates(["employee_id", "timestamp"])
    .sort_values(["employee_id", "timestamp"])
    .reset_index(drop=True)
)
pings["scenario"] = pings["employee_id"].map(scenario_by_employee)
pings["timestamp"] = pd.to_datetime(pings["timestamp"])

print(f"GPS pings: {len(pings):,} | employees: {pings.employee_id.nunique()} | time span: "
      f"{pings.timestamp.min():%H:%M}–{pings.timestamp.max():%H:%M}")
display(pings.head(8))
print("\nInjected scenarios")
display(pings.groupby(["employee_id", "scenario"]).size().rename("pings").reset_index())

## 4. Ground truth

`TRUTH` records what was injected, with the employee and time window. It is
used only for evaluation; the detectors never see these labels.


In [ ]:
def t(hours=0, minutes=0):
    return DAY + pd.Timedelta(hours=hours, minutes=minutes)


TRUTH = pd.DataFrame(
    [
        {"type": "impossible_travel", "employee_id": "EMP008", "start": t(10, 0), "end": t(10, 4), "note": "Sidi Bou Said jump"},
        {"type": "off_site_checkin", "employee_id": "EMP003", "start": t(8, 50), "end": t(8, 54), "note": "800 m from JOB-102"},
        {"type": "off_site_checkin", "employee_id": "EMP007", "start": t(8, 50), "end": t(9, 30), "note": "at JOB-103, assigned JOB-104"},
        {"type": "overstay", "employee_id": "EMP004", "start": t(10, 6), "end": t(10, 46), "note": "café 1.8 km from JOB-102"},
        {"type": "overstay", "employee_id": "EMP002", "start": t(10, 22), "end": t(10, 52), "note": "Gammarth, away from JOB-101"},
        {"type": "overstay", "employee_id": "EMP007", "start": t(8, 50), "end": t(9, 30), "note": "at another client's job"},
        {"type": "detour", "employee_id": "EMP006", "start": t(7, 45), "end": t(8, 50), "note": "via La Goulette"},
        {"type": "crew_separation", "employee_id": "EMP002", "start": t(10, 12), "end": t(11, 0), "note": "EMP001 / EMP002 pair"},
        {"type": "crew_separation", "employee_id": "EMP004", "start": t(10, 10), "end": t(10, 46), "note": "EMP003 / EMP004 pair"},
        {"type": "crew_separation", "employee_id": "EMP007", "start": t(8, 50), "end": t(9, 30), "note": "EMP007 / EMP008 pair"},
        {"type": "buddy_punch", "employee_id": "EMP007", "start": t(8, 50), "end": t(9, 30), "note": "co-located with JOB-103 crew"},
        {"type": "gps_accuracy", "employee_id": "EMP005", "start": t(11, 0), "end": t(11, 12), "note": "250 m accuracy"},
    ]
)

print("Ground-truth anomalies")
display(TRUTH.style.hide(axis="index"))

## 5. Detectors

All detectors are transparent geometry/rule-based logic, which is appropriate
for an auditable HR feature:

| Finding | Rule |
|---|---|
| `impossible_travel` | > 500 m between pings and > 150 km/h, ignoring low-accuracy fixes |
| `off_site_checkin` | first ping after 08:50 is > 300 m from the assigned job |
| `overstay` | ≥ 25 min inside an 80 m cluster more than 300 m from the job |
| `detour` | path/direct ratio ≥ 1.8 and ≥ 2 km extra before arrival |
| `crew_separation` | assigned pair > 800 m apart for ≥ 20 min |
| `buddy_punch` | different-job employees within 60 m for ≥ 10 min, at least one away from their own job |
| `gps_accuracy` | accuracy > 150 m for ≥ 5 min |

Each finding gets a risk score (0–100) and a severity. The evidence keeps the
points needed for the map popups.


In [ ]:
GEOFENCE_M = 300
DWELL_RADIUS_M = 80
DWELL_MIN_MIN = 25
MAX_SPEED_KMH = 150
DETOUR_RATIO = 1.8
DETOUR_EXTRA_M = 2_000
CREW_SEPARATION_M = 800
CREW_SEPARATION_MIN = 20

POIS = [
    ("Café El Medina", 36.8485, 10.3040),
    ("Gammarth", 36.9186, 10.2892),
    ("Tunis Centre", 36.8065, 10.1815),
    ("Sidi Bou Saïd", 36.8708, 10.3417),
    ("La Goulette", 36.8181, 10.3050),
]


def nearest_place(lat, lng, max_m=250):
    best = None
    for name, plat, plng in POIS:
        distance = haversine_m(lat, lng, plat, plng)
        if best is None or distance < best[1]:
            best = (name, distance)
    if best and best[1] <= max_m:
        return best[0]
    return "a non-client location"


def risk_and_severity(finding_type, evidence):
    if finding_type == "impossible_travel":
        score = 80 + evidence.get("max_speed_kmh", 0) / 20
    elif finding_type == "buddy_punch":
        score = 85 + evidence.get("duration_minutes", 0) / 10
    elif finding_type == "off_site_checkin":
        score = 70 + evidence.get("distance_m", 0) / 100
    elif finding_type == "overstay":
        score = 50 + evidence.get("duration_minutes", 0)
    elif finding_type == "detour":
        score = 45 + evidence.get("ratio", 0) * 10
    elif finding_type == "crew_separation":
        score = 40 + evidence.get("max_distance_m", 0) / 100 + evidence.get("duration_minutes", 0) * 0.3
    else:
        score = 20 + evidence.get("max_accuracy_m", 0) / 5
    score = int(round(min(100, max(0, score))))
    severity = "high" if score >= 75 else "medium" if score >= 45 else "low"
    return score, severity


def add_finding(findings, finding_type, employee_ids, start, end, lat, lng, summary, points=None, **evidence):
    primary = employee_ids[0]
    score, severity = risk_and_severity(finding_type, evidence)
    findings.append(
        {
            "finding_id": f"GEO-{len(findings) + 1:04d}",
            "type": finding_type,
            "employee_ids": employee_ids,
            "employee_id": primary,
            "employee_name": EMP_BY_ID.loc[primary, "name"],
            "job_id": EMP_BY_ID.loc[primary, "job_id"],
            "start": start,
            "end": end,
            "duration_min": round((end - start).total_seconds() / 60, 1),
            "lat": lat,
            "lng": lng,
            "risk_score": score,
            "severity": severity,
            "summary": summary,
            "points": points or [[lat, lng]],
            **evidence,
        }
    )


def detect_impossible_travel(pings):
    findings = []
    for emp_id, group in pings.groupby("employee_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        events = []
        for i in range(1, len(group)):
            a, b = group.iloc[i - 1], group.iloc[i]
            hours = (b.timestamp - a.timestamp).total_seconds() / 3600
            if hours <= 0:
                continue
            if a.accuracy_m > 150 or b.accuracy_m > 150:
                continue  # unreliable fix: do not label as impossible travel
            distance = haversine_m(a.lat, a.lng, b.lat, b.lng)
            speed = (distance / 1000) / hours
            if distance > 500 and speed > MAX_SPEED_KMH:
                events.append(
                    {
                        "start": a.timestamp,
                        "end": b.timestamp,
                        "points": [[a.lat, a.lng], [b.lat, b.lng]],
                        "max_speed_kmh": speed,
                        "distance_m": distance,
                    }
                )

        # Merge consecutive jumps into one event.
        merged = []
        for event in events:
            if merged and (event["start"] - merged[-1]["end"]).total_seconds() / 60 <= 15:
                merged[-1]["end"] = event["end"]
                for point in event["points"][1:]:
                    if point != merged[-1]["points"][-1]:
                        merged[-1]["points"].append(point)
                merged[-1]["max_speed_kmh"] = max(merged[-1]["max_speed_kmh"], event["max_speed_kmh"])
                merged[-1]["distance_m"] += event["distance_m"]
            else:
                merged.append(event.copy())

        for event in merged:
            add_finding(
                findings,
                "impossible_travel",
                [emp_id],
                event["start"],
                event["end"],
                event["points"][-1][0],
                event["points"][-1][1],
                f"{EMP_BY_ID.loc[emp_id, 'name']} moved {event['distance_m']/1000:.1f} km in "
                f"{max(1, (event['end']-event['start']).total_seconds()/60):.0f} min "
                f"({event['max_speed_kmh']:.0f} km/h).",
                points=event["points"],
                max_speed_kmh=round(event["max_speed_kmh"], 1),
                distance_m=round(event["distance_m"], 1),
            )
    return findings


def detect_off_site_checkin(pings):
    findings = []
    for emp_id, group in pings.groupby("employee_id"):
        group = group.sort_values("timestamp")
        job = JOB_BY_ID.loc[EMP_BY_ID.loc[emp_id, "job_id"]]
        candidates = group[group.timestamp >= DAY + pd.Timedelta(hours=8, minutes=50)]
        if candidates.empty:
            continue
        first = candidates.iloc[0]
        distance = haversine_m(first.lat, first.lng, job.lat, job.lng)
        if distance > GEOFENCE_M:
            add_finding(
                findings,
                "off_site_checkin",
                [emp_id],
                first.timestamp,
                first.timestamp + pd.Timedelta(minutes=4),
                first.lat,
                first.lng,
                f"{EMP_BY_ID.loc[emp_id, 'name']} checked in {distance:,.0f} m from "
                f"{job.area} (geofence {GEOFENCE_M} m).",
                points=[[first.lat, first.lng], [job.lat, job.lng]],
                distance_m=round(distance, 1),
                job_lat=float(job.lat),
                job_lng=float(job.lng),
            )
    return findings


def detect_overstays(pings):
    findings = []
    for emp_id, group in pings.groupby("employee_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        job = JOB_BY_ID.loc[EMP_BY_ID.loc[emp_id, "job_id"]]
        clusters = []
        current = [group.iloc[0]]
        for i in range(1, len(group)):
            row = group.iloc[i]
            center_lat = float(np.mean([p.lat for p in current]))
            center_lng = float(np.mean([p.lng for p in current]))
            if haversine_m(row.lat, row.lng, center_lat, center_lng) <= DWELL_RADIUS_M:
                current.append(row)
            else:
                clusters.append(current)
                current = [row]
        clusters.append(current)

        for cluster in clusters:
            start, end = cluster[0].timestamp, cluster[-1].timestamp
            duration = (end - start).total_seconds() / 60
            if duration < DWELL_MIN_MIN:
                continue
            lat = float(np.mean([p.lat for p in cluster]))
            lng = float(np.mean([p.lng for p in cluster]))
            distance_to_job = haversine_m(lat, lng, job.lat, job.lng)
            if distance_to_job <= GEOFENCE_M:
                continue  # staying at the client's house is the job
            place = nearest_place(lat, lng)
            add_finding(
                findings,
                "overstay",
                [emp_id],
                start,
                end,
                lat,
                lng,
                f"{EMP_BY_ID.loc[emp_id, 'name']} stayed {duration:.0f} min at {place}, "
                f"{distance_to_job/1000:.1f} km from {job.area}.",
                points=[[p.lat, p.lng] for p in cluster[:: max(1, len(cluster)//12)]],
                duration_minutes=round(duration, 1),
                distance_to_job_m=round(distance_to_job, 1),
                place=place,
            )
    return findings


def detect_detours(pings):
    findings = []
    for emp_id, group in pings.groupby("employee_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        job = JOB_BY_ID.loc[EMP_BY_ID.loc[emp_id, "job_id"]]
        before = group[group.timestamp <= DAY + pd.Timedelta(hours=9, minutes=30)]
        arrival_index = None
        for i, row in before.iterrows():
            if haversine_m(row.lat, row.lng, job.lat, job.lng) <= 120:
                arrival_index = i
                break
        if arrival_index is None or arrival_index < 3:
            continue
        path = before.loc[:arrival_index]
        path_length = sum(
            haversine_m(path.iloc[i - 1].lat, path.iloc[i - 1].lng, path.iloc[i].lat, path.iloc[i].lng)
            for i in range(1, len(path))
        )
        direct = haversine_m(path.iloc[0].lat, path.iloc[0].lng, path.iloc[-1].lat, path.iloc[-1].lng)
        if direct <= 0:
            continue
        ratio = path_length / direct
        if ratio >= DETOUR_RATIO and path_length - direct >= DETOUR_EXTRA_M:
            add_finding(
                findings,
                "detour",
                [emp_id],
                path.iloc[0].timestamp,
                path.iloc[-1].timestamp,
                path.iloc[-1].lat,
                path.iloc[-1].lng,
                f"{EMP_BY_ID.loc[emp_id, 'name']} drove {path_length/1000:.1f} km for a "
                f"{direct/1000:.1f} km trip ({ratio:.1f}× direct).",
                points=[[row.lat, row.lng] for _, row in path.iterrows()],
                actual_m=round(path_length, 1),
                direct_m=round(direct, 1),
                ratio=round(ratio, 2),
            )
    return findings


def _time_bucket(ts, minutes=10):
    return ts.floor(f"{minutes}min")


def detect_crew_separation(pings):
    findings = []
    for job_id, crew in CREWS.items():
        if len(crew) < 2:
            continue
        job = JOB_BY_ID.loc[job_id]
        window = pings[
            (pings.job_id == job_id)
            & (pings.timestamp >= DAY + pd.Timedelta(hours=8, minutes=50))
            & (pings.timestamp <= DAY + pd.Timedelta(hours=13))
        ]
        for i in range(len(crew)):
            for j in range(i + 1, len(crew)):
                a_id, b_id = crew[i], crew[j]
                a = window[window.employee_id == a_id].set_index(window[window.employee_id == a_id].timestamp.apply(_time_bucket))
                b = window[window.employee_id == b_id].set_index(window[window.employee_id == b_id].timestamp.apply(_time_bucket))
                buckets = sorted(set(a.index) & set(b.index))
                separated = []
                for bucket in buckets:
                    pa = a.loc[[bucket]].iloc[0]
                    pb = b.loc[[bucket]].iloc[0]
                    distance = haversine_m(pa.lat, pa.lng, pb.lat, pb.lng)
                    if distance > CREW_SEPARATION_M:
                        separated.append((bucket, distance, pa, pb))

                if not separated:
                    continue

                groups = []
                current = [separated[0]]
                for item in separated[1:]:
                    if (item[0] - current[-1][0]).total_seconds() / 60 <= 10:
                        current.append(item)
                    else:
                        groups.append(current)
                        current = [item]
                groups.append(current)

                for group in groups:
                    duration = (group[-1][0] - group[0][0]).total_seconds() / 60 + 10
                    if duration < CREW_SEPARATION_MIN:
                        continue
                    max_item = max(group, key=lambda item: item[1])
                    pa, pb = max_item[2], max_item[3]
                    add_finding(
                        findings,
                        "crew_separation",
                        [a_id, b_id],
                        group[0][0],
                        group[-1][0] + pd.Timedelta(minutes=10),
                        (pa.lat + pb.lat) / 2,
                        (pa.lng + pb.lng) / 2,
                        f"{EMP_BY_ID.loc[a_id, 'name']} and {EMP_BY_ID.loc[b_id, 'name']} were "
                        f"{max_item[1]/1000:.1f} km apart for ~{duration:.0f} min during {job.area}.",
                        points=[[pa.lat, pa.lng], [pb.lat, pb.lng]],
                        max_distance_m=round(max_item[1], 1),
                        duration_minutes=round(duration, 1),
                        employee_names=[EMP_BY_ID.loc[a_id, "name"], EMP_BY_ID.loc[b_id, "name"]],
                    )
    return findings


def detect_buddy_punch(pings):
    """Two employees assigned to different jobs, co-located for 10+ minutes."""
    findings = []
    job_windows = pings[
        (pings.timestamp >= DAY + pd.Timedelta(hours=8, minutes=45))
        & (pings.timestamp <= DAY + pd.Timedelta(hours=12))
    ].copy()
    job_windows["bucket"] = job_windows.timestamp.apply(_time_bucket)

    by_bucket = {bucket: group for bucket, group in job_windows.groupby("bucket")}
    pair_events = []
    emp_ids = sorted(job_windows.employee_id.unique())
    for i in range(len(emp_ids)):
        for j in range(i + 1, len(emp_ids)):
            a_id, b_id = emp_ids[i], emp_ids[j]
            a_job = EMP_BY_ID.loc[a_id, "job_id"]
            b_job = EMP_BY_ID.loc[b_id, "job_id"]
            if a_job == b_job:
                continue
            job_a = JOB_BY_ID.loc[a_job]
            job_b = JOB_BY_ID.loc[b_job]

            matches = []
            for bucket in sorted(by_bucket):
                ga = by_bucket[bucket]
                ga = ga[ga.employee_id == a_id]
                gb = by_bucket[bucket]
                gb = gb[gb.employee_id == b_id]
                if ga.empty or gb.empty:
                    continue
                pa, pb = ga.iloc[0], gb.iloc[0]
                distance = haversine_m(pa.lat, pa.lng, pb.lat, pb.lng)
                if distance > 60:
                    continue
                a_away = haversine_m(pa.lat, pa.lng, job_a.lat, job_a.lng) > GEOFENCE_M
                b_away = haversine_m(pb.lat, pb.lng, job_b.lat, job_b.lng) > GEOFENCE_M
                if not (a_away or b_away):
                    continue
                matches.append((bucket, pa, pb, a_away, b_away, distance))

            if not matches:
                continue

            groups = []
            current = [matches[0]]
            for item in matches[1:]:
                if (item[0] - current[-1][0]).total_seconds() / 60 <= 10:
                    current.append(item)
                else:
                    groups.append(current)
                    current = [item]
            groups.append(current)

            for group in groups:
                duration = (group[-1][0] - group[0][0]).total_seconds() / 60 + 10
                if duration < 10:
                    continue
                pa, pb = group[0][1], group[0][2]
                away_employee = a_id if group[0][3] else b_id
                own_job = a_job if group[0][3] else b_job
                pair_events.append(
                    {
                        "start": group[0][0],
                        "end": group[-1][0] + pd.Timedelta(minutes=10),
                        "employee_ids": [a_id, b_id],
                        "away_employee": away_employee,
                        "own_job": own_job,
                        "lat": (pa.lat + pb.lat) / 2,
                        "lng": (pa.lng + pb.lng) / 2,
                        "duration": duration,
                    }
                )

    # Merge overlapping pair events at the same place (e.g. Grace + both JOB-103 crew).
    merged = []
    for event in pair_events:
        target = None
        for existing in merged:
            if (
                event["start"] <= existing["end"] + pd.Timedelta(minutes=10)
                and event["end"] >= existing["start"] - pd.Timedelta(minutes=10)
                and haversine_m(event["lat"], event["lng"], existing["lat"], existing["lng"]) < 100
            ):
                target = existing
                break
        if target:
            target["end"] = max(target["end"], event["end"])
            target["employee_ids"] = sorted(set(target["employee_ids"] + event["employee_ids"]))
            target["duration"] = (target["end"] - target["start"]).total_seconds() / 60
        else:
            merged.append(event.copy())

    for event in merged:
        away_name = EMP_BY_ID.loc[event["away_employee"], "name"]
        own_area = JOB_BY_ID.loc[event["own_job"], "area"]
        employee_order = [event["away_employee"]] + [
            emp for emp in event["employee_ids"] if emp != event["away_employee"]
        ]
        add_finding(
            findings,
            "buddy_punch",
            employee_order,
            event["start"],
            event["end"],
            event["lat"],
            event["lng"],
            f"{away_name} was co-located with another crew for {event['duration']:.0f} min "
            f"while assigned to {own_area}.",
            points=[[event["lat"], event["lng"]]],
            duration_minutes=round(event["duration"], 1),
            away_employee=event["away_employee"],
            own_job=event["own_job"],
        )
    return findings


def detect_gps_accuracy(pings, threshold_m=150):
    findings = []
    for emp_id, group in pings.groupby("employee_id"):
        group = group.sort_values("timestamp").reset_index(drop=True)
        bad = group[group.accuracy_m > threshold_m]
        if bad.empty:
            continue
        groups = []
        current = [bad.iloc[0]]
        for i in range(1, len(bad)):
            row = bad.iloc[i]
            if (row.timestamp - current[-1].timestamp).total_seconds() / 60 <= 10:
                current.append(row)
            else:
                groups.append(current)
                current = [row]
        groups.append(current)
        for group_bad in groups:
            bad_frame = pd.DataFrame(group_bad)
            start, end = bad_frame.iloc[0].timestamp, bad_frame.iloc[-1].timestamp
            duration = (end - start).total_seconds() / 60
            if duration < 5:
                continue
            lat = float(bad_frame.lat.mean())
            lng = float(bad_frame.lng.mean())
            add_finding(
                findings,
                "gps_accuracy",
                [emp_id],
                start,
                end,
                lat,
                lng,
                f"{EMP_BY_ID.loc[emp_id, 'name']} had {bad_frame.accuracy_m.max():.0f} m GPS accuracy "
                f"for {duration:.0f} min — location evidence is unreliable.",
                points=[[row.lat, row.lng] for _, row in bad_frame.iterrows()],
                max_accuracy_m=round(float(bad_frame.accuracy_m.max()), 1),
                pings_affected=len(bad_frame),
            )
    return findings


findings = []
findings += detect_impossible_travel(pings)
findings += detect_off_site_checkin(pings)
findings += detect_overstays(pings)
findings += detect_detours(pings)
findings += detect_crew_separation(pings)
findings += detect_buddy_punch(pings)
findings += detect_gps_accuracy(pings)

findings = sorted(findings, key=lambda f: (-f["risk_score"], f["start"]))
for index, finding in enumerate(findings, start=1):
    finding["finding_id"] = f"GEO-{index:04d}"

findings_df = pd.DataFrame(findings)
print(f"Detected findings: {len(findings)}")
display(
    findings_df[
        ["finding_id", "type", "employee_id", "severity", "risk_score", "duration_min", "summary"]
    ]
)

## 6. Employee risk roll-up

HR should see a person-level queue, not just raw events. The roll-up combines
the highest finding score with a small penalty for additional findings.


In [ ]:
rows = []
for emp_id, group in findings_df.groupby("employee_id"):
    types = group.type.tolist()
    score = min(100, group.risk_score.max() + 5 * (len(group) - 1))
    rows.append(
        {
            "employee_id": emp_id,
            "employee_name": EMP_BY_ID.loc[emp_id, "name"],
            "job_id": EMP_BY_ID.loc[emp_id, "job_id"],
            "findings": len(group),
            "high": int((group.severity == "high").sum()),
            "medium": int((group.severity == "medium").sum()),
            "low": int((group.severity == "low").sum()),
            "risk_score": int(score),
            "types": ", ".join(sorted(set(types))),
        }
    )

risk_df = pd.DataFrame(rows).sort_values(["risk_score", "findings"], ascending=False)
print("Employee risk roll-up")
display(risk_df.style.background_gradient(cmap="Reds", subset=["risk_score"]).hide(axis="index"))

## 7. Evaluation against injected ground truth

A finding is a true positive when its type, employee, and time window overlap a
ground-truth event. Because the anomalies are deliberately injected into a
controlled synthetic day, perfect recovery here validates the geometry and the
labels; it is **not** a claim of real-world precision. The threshold sweep in
the next section shows how quickly extra events appear when rules are loosened.


In [ ]:
def event_overlaps(finding, truth, tolerance_min=20):
    tolerance = pd.Timedelta(minutes=tolerance_min)
    return finding["start"] <= truth["end"] + tolerance and finding["end"] >= truth["start"] - tolerance


matched_truth = set()
matched_findings = set()
matches = []

for ti, truth in TRUTH.iterrows():
    for fi, finding in enumerate(findings):
        if fi in matched_findings or ti in matched_truth:
            continue
        if finding["type"] != truth["type"]:
            continue
        if truth["employee_id"] not in finding["employee_ids"]:
            continue
        if event_overlaps(finding, truth):
            matched_truth.add(ti)
            matched_findings.add(fi)
            matches.append(
                {
                    "truth": truth["type"],
                    "employee_id": truth["employee_id"],
                    "finding_id": finding["finding_id"],
                    "risk_score": finding["risk_score"],
                }
            )
            break

evaluation_rows = []
for finding_type in sorted(TRUTH.type.unique()):
    truth_total = int((TRUTH.type == finding_type).sum())
    found_total = int((findings_df.type == finding_type).sum())
    tp = int(
        sum(
            1
            for ti, truth in TRUTH[TRUTH.type == finding_type].iterrows()
            if ti in matched_truth
        )
    )
    fp = found_total - tp
    fn = truth_total - tp
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    evaluation_rows.append(
        {
            "type": finding_type,
            "truth": truth_total,
            "detected": found_total,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": round(precision, 3),
            "recall": round(recall, 3),
            "f1": round(f1, 3),
        }
    )

evaluation_df = pd.DataFrame(evaluation_rows)
total_tp = int(evaluation_df.tp.sum())
total_fp = int(evaluation_df.fp.sum())
total_fn = int(evaluation_df.fn.sum())
print("Per-type evaluation")
display(evaluation_df.style.hide(axis="index"))
print(
    f"Micro precision={total_tp/(total_tp+total_fp):.3f}  "
    f"recall={total_tp/(total_tp+total_fn):.3f}  "
    f"F1={2*total_tp/(2*total_tp+total_fp+total_fn):.3f}"
)

fig, ax = plt.subplots(figsize=(10, 3.5))
x = np.arange(len(evaluation_df))
width = 0.26
ax.bar(x - width, evaluation_df.tp, width, label="TP", color="#111111")
ax.bar(x, evaluation_df.fp, width, label="FP", color="#e61919")
ax.bar(x + width, evaluation_df.fn, width, label="FN", color="#bbbbbb")
ax.set_xticks(x)
ax.set_xticklabels([t.replace("_", "\n") for t in evaluation_df.type], fontsize=9)
ax.set_ylabel("Events")
ax.set_title("Detection quality by anomaly type (synthetic ground truth)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "geo_fraud_evaluation.png", dpi=160)
plt.show()

## 7b. Threshold sensitivity

The left panel is the dwell threshold. At 10 minutes the legitimate coffee stop
is flagged; at 15 minutes and above it is ignored. The right panel is the
detour ratio: the injected detour is 2.6× direct, so it survives until the
threshold passes 2.6. This is how the policy values were selected.


In [ ]:
dwell_sweep = []
for threshold in [10, 15, 20, 25, 30, 40, 60]:
    globals()["DWELL_MIN_MIN"] = threshold
    dwell_sweep.append({"threshold_min": threshold, "findings": len(detect_overstays(pings))})
globals()["DWELL_MIN_MIN"] = 25

detour_sweep = []
for threshold in [1.3, 1.5, 1.7, 1.8, 2.0, 2.3, 2.6]:
    globals()["DETOUR_RATIO"] = threshold
    detour_sweep.append({"ratio": threshold, "findings": len(detect_detours(pings))})
globals()["DETOUR_RATIO"] = 1.8

dwell_sweep_df = pd.DataFrame(dwell_sweep)
detour_sweep_df = pd.DataFrame(detour_sweep)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(dwell_sweep_df.threshold_min, dwell_sweep_df.findings, marker="o", color="#e61919")
axes[0].axvline(25, color="#111", linestyle="--", linewidth=1)
axes[0].set_xlabel("Minimum dwell (minutes)")
axes[0].set_ylabel("Overstay findings")
axes[0].set_title("Dwell threshold")
axes[0].grid(alpha=0.2)
axes[1].plot(detour_sweep_df.ratio, detour_sweep_df.findings, marker="o", color="#ef6c00")
axes[1].axvline(1.8, color="#111", linestyle="--", linewidth=1)
axes[1].set_xlabel("Detour ratio (actual / direct)")
axes[1].set_ylabel("Detour findings")
axes[1].set_title("Detour threshold")
axes[1].grid(alpha=0.2)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "geo_fraud_thresholds.png", dpi=160)
plt.show()

print("Dwell sweep")
display(dwell_sweep_df)
print("Detour sweep")
display(detour_sweep_df)

## 8. Static map

The grey lines are all employee routes. Black squares are client sites with a
300 m geofence. Red translucent circles mark overstays away from a client;
red dashed lines join separated crew members; orange is the detour. The blue
map is saved next to this notebook as `geo_fraud_static_map.png`.


In [ ]:
lat0, lng0 = 36.855, 10.255
cos_lat = math.cos(math.radians(lat0))

def to_xy(lat, lng):
    return (lng - lng0) * cos_lat, lat - lat0


fig, ax = plt.subplots(figsize=(11, 9))

# Employee routes first (grey context).
for emp_id, group in pings.groupby("employee_id"):
    group = group.sort_values("timestamp")
    xs, ys = to_xy(group.lat.values, group.lng.values)
    ax.plot(xs, ys, color="#b7b7b7", linewidth=1.0, alpha=0.85, zorder=1)

# Job sites + geofence.
for _, job in JOBS.iterrows():
    x, y = to_xy(job.lat, job.lng)
    ax.scatter(x, y, marker="s", s=90, color="#111111", zorder=6)
    ax.add_patch(Circle((x, y), GEOFENCE_M / 111_320, color="#111111", fill=False, linewidth=1.0, linestyle=":", zorder=2))
    ax.annotate(f"{job.client}\n{job.area}", (x, y), textcoords="offset points", xytext=(7, 7), fontsize=8, fontweight="bold")

# Findings.
for finding in findings:
    kind = finding["type"]
    x, y = to_xy(finding["lat"], finding["lng"])
    if kind == "overstay":
        radius = min(450, 100 + finding.get("duration_minutes", 30) * 3) / 111_320
        ax.add_patch(Circle((x, y), radius, color="#e61919", alpha=0.25, zorder=3))
        ax.scatter(x, y, s=50, color="#e61919", edgecolor="white", linewidth=1.2, zorder=7)
        ax.annotate(f"overstay\n{finding['duration_min']:.0f} min", (x, y), textcoords="offset points", xytext=(8, -14), fontsize=7, color="#a00000")
    elif kind == "crew_separation":
        p1, p2 = finding["points"][0], finding["points"][1]
        x1, y1 = to_xy(*p1)
        x2, y2 = to_xy(*p2)
        ax.plot([x1, x2], [y1, y2], color="#e61919", linewidth=2.2, linestyle="--", zorder=5)
        ax.scatter([x1, x2], [y1, y2], s=30, color="#e61919", zorder=6)
        ax.annotate(f"separated\n{finding['max_distance_m']/1000:.1f} km", ((x1+x2)/2, (y1+y2)/2), textcoords="offset points", xytext=(6, 6), fontsize=7, color="#a00000")
    elif kind == "detour":
        xs = [(p[1] - lng0) * cos_lat for p in finding["points"]]
        ys = [p[0] - lat0 for p in finding["points"]]
        ax.plot(xs, ys, color="#f07c00", linewidth=2.2, zorder=5)
    elif kind == "impossible_travel":
        for index in range(1, len(finding["points"])):
            x1, y1 = to_xy(*finding["points"][index - 1])
            x2, y2 = to_xy(*finding["points"][index])
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#e61919", lw=2.0, linestyle="--"))
            ax.scatter([x2], [y2], marker="o", s=45, color="#e61919", zorder=7)
    elif kind == "off_site_checkin":
        ax.scatter(x, y, marker="*", s=160, color="#e61919", edgecolor="white", linewidth=1.0, zorder=8)
    elif kind == "buddy_punch":
        ax.scatter(x, y, marker="X", s=140, color="#7a00a8", edgecolor="white", linewidth=1.2, zorder=8)
    elif kind == "gps_accuracy":
        ax.scatter(x, y, marker="o", s=120, facecolors="none", edgecolors="#777777", linewidth=1.2, zorder=6)

ax.set_title("One synthetic day — detected geo-fraud in Greater Tunis", fontsize=14, fontweight="bold", loc="left")
ax.set_xlabel("Longitude offset")
ax.set_ylabel("Latitude offset")
ax.set_aspect("equal")
ax.grid(alpha=0.15)
legend_handles = [
    Line2D([0], [0], marker="s", color="w", markerfacecolor="#111111", markersize=8, label="Client job site"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e61919", markersize=8, label="Overstay (red zone)"),
    Line2D([0], [0], color="#e61919", linestyle="--", linewidth=2, label="Crew separated"),
    Line2D([0], [0], color="#b71c1c", linewidth=2.2, label="Impossible travel"),
    Line2D([0], [0], color="#f07c00", linewidth=2.2, label="Detour"),
    Line2D([0], [0], marker="X", color="w", markerfacecolor="#7a00a8", markersize=9, label="Buddy punch"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="#e61919", markersize=12, label="Off-site check-in"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="none", markeredgecolor="#777777", markersize=9, label="Poor GPS accuracy"),
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=8, frameon=True)
fig.tight_layout()
fig.savefig(OUT / "geo_fraud_static_map.png", dpi=180)
plt.show()

print("Saved:", OUT / "geo_fraud_static_map.png")

## 9. Interactive Folium map

The interactive map has one layer per finding type, plus route and job layers.
Red circles are overstay zones (sized by duration), red dashed lines are crew
separations, orange lines are detours, and red/purple markers cover impossible
travel, buddy punching, and off-site check-ins.

Folium basemap tiles need Internet. If tiles are blank, the shapes and popups
still work; the static PNG above is fully offline. The HTML is saved to
`/kaggle/working/geo_fraud_output/geo_fraud_map.html`.


In [ ]:
m = folium.Map(location=[36.862, 10.255], zoom_start=12, tiles="CartoDB positron", control_scale=True)

# Job sites and geofences.
jobs_layer = folium.FeatureGroup(name="Client job sites", show=True)
for _, job in JOBS.iterrows():
    folium.Marker(
        [job.lat, job.lng],
        tooltip=f"{job.client} — {job.area}",
        popup=folium.Popup(
            f"<b>{job.client}</b><br>{job.area}<br>{job.start:%H:%M}–{job.end:%H:%M}",
            max_width=250,
        ),
        icon=folium.Icon(color="green", icon="home", prefix="fa"),
    ).add_to(jobs_layer)
    folium.Circle(
        [job.lat, job.lng],
        radius=GEOFENCE_M,
        color="#2e7d32",
        weight=1,
        dash_array="5 5",
        fill=False,
        tooltip=f"Geofence {GEOFENCE_M} m",
    ).add_to(jobs_layer)
jobs_layer.add_to(m)

# All routes as thin context lines, grouped by employee.
routes_layer = folium.FeatureGroup(name="Employee routes", show=True)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#17becf"]
for index, (emp_id, group) in enumerate(pings.groupby("employee_id")):
    group = group.sort_values("timestamp")
    folium.PolyLine(
        [[row.lat, row.lng] for _, row in group.iterrows()],
        color=colors[index % len(colors)],
        weight=2,
        opacity=0.55,
        tooltip=f"{EMP_BY_ID.loc[emp_id, 'name']} ({emp_id})",
    ).add_to(routes_layer)
routes_layer.add_to(m)

# Findings, one layer per type.
layers = {
    "Overstay at non-client location": folium.FeatureGroup(name="Overstay (red zones)", show=True),
    "Crew separation": folium.FeatureGroup(name="Crew separation", show=True),
    "Detour": folium.FeatureGroup(name="Route detour", show=True),
    "Impossible travel": folium.FeatureGroup(name="Impossible travel", show=True),
    "Buddy punch": folium.FeatureGroup(name="Buddy punch", show=True),
    "Off-site check-in": folium.FeatureGroup(name="Off-site check-in", show=True),
    "GPS accuracy": folium.FeatureGroup(name="Poor GPS accuracy", show=False),
}

for finding in findings:
    kind = finding["type"]
    popup_html = (
        f"<b>{finding['finding_id']} · {kind.replace('_', ' ').title()}</b><br>"
        f"Risk: <b>{finding['risk_score']}/100</b> ({finding['severity']})<br>"
        f"Employee(s): {', '.join(finding['employee_ids'])}<br>"
        f"{finding['summary']}<br>"
        f"<small>{finding['start']:%H:%M}–{finding['end']:%H:%M}</small>"
    )
    popup = folium.Popup(popup_html, max_width=360)

    if kind == "overstay":
        radius = min(450, 100 + finding.get("duration_minutes", 30) * 3)
        layer = layers["Overstay at non-client location"]
        folium.Circle(
            [finding["lat"], finding["lng"]],
            radius=radius,
            color="#c62828",
            weight=1,
            fill=True,
            fill_color="#e61919",
            fill_opacity=0.22,
            popup=popup,
            tooltip=f"Overstay {finding['duration_min']:.0f} min",
        ).add_to(layer)
        folium.CircleMarker(
            [finding["lat"], finding["lng"]],
            radius=5,
            color="white",
            weight=1,
            fill=True,
            fill_color="#e61919",
            fill_opacity=1,
            popup=popup,
        ).add_to(layer)
    elif kind == "crew_separation":
        layer = layers["Crew separation"]
        folium.PolyLine(
            finding["points"],
            color="#c62828",
            weight=3,
            dash_array="9 7",
            popup=popup,
            tooltip="Crew separated",
        ).add_to(layer)
        for point in finding["points"]:
            folium.CircleMarker(point, radius=4, color="#c62828", fill=True, fill_opacity=1).add_to(layer)
    elif kind == "detour":
        layer = layers["Detour"]
        folium.PolyLine(
            finding["points"],
            color="#ef6c00",
            weight=4,
            popup=popup,
            tooltip=f"Detour: {finding['ratio']}× direct",
        ).add_to(layer)
    elif kind == "impossible_travel":
        layer = layers["Impossible travel"]
        folium.PolyLine(
            finding["points"],
            color="#b71c1c",
            weight=4,
            dash_array="4 6",
            popup=popup,
            tooltip="Impossible travel",
        ).add_to(layer)
        folium.Marker(
            finding["points"][-1],
            icon=folium.Icon(color="red", icon="bolt", prefix="fa"),
            popup=popup,
        ).add_to(layer)
    elif kind == "buddy_punch":
        layer = layers["Buddy punch"]
        folium.Marker(
            [finding["lat"], finding["lng"]],
            icon=folium.Icon(color="purple", icon="user", prefix="fa"),
            popup=popup,
            tooltip="Buddy punch suspected",
        ).add_to(layer)
    elif kind == "off_site_checkin":
        layer = layers["Off-site check-in"]
        folium.Marker(
            [finding["lat"], finding["lng"]],
            icon=folium.Icon(color="red", icon="flag", prefix="fa"),
            popup=popup,
            tooltip="Off-site check-in",
        ).add_to(layer)
        if len(finding["points"]) > 1:
            folium.PolyLine(finding["points"], color="#c62828", weight=2, dash_array="3 5", popup=popup).add_to(layer)
    elif kind == "gps_accuracy":
        layer = layers["GPS accuracy"]
        folium.Circle(
            [finding["lat"], finding["lng"]],
            radius=160,
            color="#616161",
            weight=1,
            dash_array="4 4",
            fill=True,
            fill_color="#9e9e9e",
            fill_opacity=0.18,
            popup=popup,
            tooltip="Unreliable GPS",
        ).add_to(layer)

for layer in layers.values():
    layer.add_to(m)

# Simple legend.
legend = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: #f7f6f2; border: 1px solid #111; padding: 12px 14px;
            font: 12px/1.6 Arial, sans-serif; color: #111; max-width: 260px;">
  <b>MOVEUP · GEO-FRAUD</b><br>
  <span style="color:#2e7d32;">■</span> Client job site + 300 m geofence<br>
  <span style="color:#e61919;">●</span> Red zone = overstay away from client<br>
  <span style="color:#c62828;">━</span> Dashed line = crew separated<br>
  <span style="color:#ef6c00;">━</span> Orange = longer route / detour<br>
  <span style="color:#b71c1c;">➤</span> Impossible travel<br>
  <span style="color:#7a00a8;">✕</span> Buddy punch suspicion<br>
  <span style="color:#616161;">◌</span> Poor GPS accuracy
</div>
"""
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(m)

map_path = OUT / "geo_fraud_map.html"
m.save(str(map_path))
print("Interactive map saved:", map_path)
display(m)

## 10. Export findings and policy

The CSV/JSON outputs are deliberately shaped for the Next.js app:

- `geo_findings.json` — full evidence, including map points
- `geo_findings.csv` — HR review table
- `employee_risk.csv` — person-level queue
- `evaluation.csv` / `ground_truth.csv` — reproducibility
- `geo_fraud_policy.json` — thresholds to store server-side

For production, ingest real `TrackingPing` documents, replace the synthetic
generator, and keep the policy thresholds configurable per city/client.


In [ ]:
def json_ready(value):
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, list):
        return [json_ready(item) for item in value]
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    return value


export_findings = []
for finding in findings:
    item = {key: value for key, value in finding.items()}
    export_findings.append(json_ready(item))

with open(OUT / "geo_findings.json", "w", encoding="utf-8") as handle:
    json.dump(export_findings, handle, indent=2, ensure_ascii=False)

findings_export = findings_df.drop(columns=["points"], errors="ignore")
findings_export.to_csv(OUT / "geo_findings.csv", index=False)
risk_df.to_csv(OUT / "employee_risk.csv", index=False)
evaluation_df.to_csv(OUT / "evaluation.csv", index=False)
TRUTH.to_csv(OUT / "ground_truth.csv", index=False)

policy = {
    "version": 1,
    "geofence_radius_m": GEOFENCE_M,
    "dwell_radius_m": DWELL_RADIUS_M,
    "dwell_min_minutes": DWELL_MIN_MIN,
    "max_speed_kmh": MAX_SPEED_KMH,
    "detour_ratio": DETOUR_RATIO,
    "detour_extra_m": DETOUR_EXTRA_M,
    "crew_separation_m": CREW_SEPARATION_M,
    "crew_separation_min_minutes": CREW_SEPARATION_MIN,
    "gps_accuracy_threshold_m": 150,
    "severity_weights": {
        "impossible_travel": 80,
        "buddy_punch": 85,
        "off_site_checkin": 70,
        "overstay": 50,
        "detour": 45,
        "crew_separation": 40,
        "gps_accuracy": 20,
    },
}
with open(OUT / "geo_fraud_policy.json", "w", encoding="utf-8") as handle:
    json.dump(policy, handle, indent=2)

print("Exported files:")
for path in sorted(OUT.iterdir()):
    print(" -", path.name, f"({path.stat().st_size / 1024:.1f} KB)")

print("\nTop 5 findings")
display(findings_export.head(5)[["finding_id", "type", "employee_id", "risk_score", "summary"]])

## Next step for MoveUp

A minimal production model is three collections: `JobSite` (lat/lng + geofence),
`TrackingPing` (employee, timestamp, lat/lng, accuracy), and `GeoFinding`
(type, employee, window, score, evidence). The detectors in this notebook can
run as a scheduled server job or an API route; the existing Leaflet component
can render the same red circles and dashed separation lines from
`geo_findings.json`.

Privacy: GPS is personal data. Collect only during shifts, tell employees what
is monitored, retain the minimum needed, and require human review before any
disciplinary action.
